In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

survival_df = pd.read_csv("TCGA_data/survival_data.txt", sep="\t")
phenotype_df = pd.read_csv("TCGA_data/survival_full_set.csv")

merged_df = survival_df.merge(
    phenotype_df,
    on="sample",
    how="inner"
)

# Split by survival status
survived = merged_df[merged_df['OS'] == 1]
not_survived = merged_df[merged_df['OS'] == 0]

RT_df = merged_df[merged_df['treatment_or_therapy.treatments.diagnoses'] == "RT"]
TMZ_df = merged_df[merged_df["treatment_or_therapy.treatments.diagnoses"] == "TMZ"]
RT_and_TMZ_df = merged_df[merged_df["treatment_or_therapy.treatments.diagnoses"] == "RT/TMZ"]
no_treatment_df = merged_df[merged_df["treatment_or_therapy.treatments.diagnoses"] == "no treatment"]

# --- Event encoding ---
merged_df["event"] = merged_df["vital_status.demographic"].map({
    "Dead": 1,
    "Alive": 0
})

# --- Clean OS.time ---
merged_df["OS.time"] = pd.to_numeric(merged_df["OS.time"], errors="coerce")

# --- Drop missing required fields ---
merged_df = merged_df.dropna(subset=["OS.time", "event", "Methylation_Status"])

# --- TMZ group classification ---
def classify_tmz(val):
    val_str = str(val).lower()
    if "tmz" in val_str or "temozolomide" in val_str:
        return "TMZ"        # catches both "TMZ" and "RT/TMZ"
    else:
        return "Non-TMZ"    # catches RT alone, no treatment, etc.

merged_df["TMZ_group"] = merged_df[
    "treatment_or_therapy.treatments.diagnoses"
].apply(classify_tmz)